# 05 · Export for the app

The trained detector from notebook 03 is prepared for deployment. It is
converted to ONNX, a portable model format that runs on an ordinary processor
without PyTorch, so the backend can score uploaded samples on a modest server.
The conversion is checked against the original model before anything else uses
it. The notebook then cuts the twelve highest-scoring tiles for each of the 65
patients, so the review screen can show the reviewer the actual image regions
the detector ranked highest.

## 1 · Setup

Installs the ONNX tools and confirms the detector, its training tiles, and the
saved patient scores from notebook 04 are attached.

In [1]:
!pip install -q onnx onnxruntime

import os, io, json, time, pathlib, zipfile, collections
import numpy as np, torch, torch.nn as nn, torchvision, onnxruntime as ort

WORK   = pathlib.Path("/kaggle/working")
DET    = "/kaggle/input/notebooks/oglorypaul/03-detector/detector.pt"
TRN    = "/kaggle/input/notebooks/oglorypaul/03-detector/trainset"
SCORES = "/kaggle/input/notebooks/oglorypaul/04-rescore/patient_scores.npz"

for p in (DET, TRN, SCORES):
    assert os.path.exists(p), f"not attached: {p}"
print(f"torch {torch.__version__} · onnxruntime {ort.__version__} · inputs attached")

torch 2.11.0+cu128 · onnxruntime 1.31.0 · inputs attached


## 2 · Converting the detector

The ResNet18 is rebuilt and its trained weights loaded, exactly as in notebook
04. Two steps that previously happened outside the model, the image
normalisation and the final sigmoid, are built into the exported model, so the
backend only has to supply tiles as numbers between 0 and 1 and gets back
probabilities. Fewer steps outside the model means fewer places for the app's
preprocessing to drift from the research.

The exported model is then checked against the original on 600 of its own
training tiles; the two must agree to within 0.0001.

In [2]:
class Exportable(nn.Module):
    """ResNet18 detector with normalisation and sigmoid built in."""
    def __init__(self, net):
        super().__init__()
        self.net = net
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):                      # x: float32 (N, 3, 256, 256), values 0–1
        return torch.sigmoid(self.net((x - self.mean) / self.std).squeeze(1))

net = torchvision.models.resnet18(weights=None)
net.fc = nn.Linear(512, 1)
net.load_state_dict(torch.load(DET, map_location="cpu"))
model = Exportable(net.eval()).eval()

ONNX = WORK / "detector.onnx"
kwargs = dict(input_names=["tiles"], output_names=["scores"],
              dynamic_axes={"tiles": {0: "n"}, "scores": {0: "n"}}, opset_version=17)
try:
    torch.onnx.export(model, torch.zeros(1, 3, 256, 256), ONNX, dynamo=False, **kwargs)
except TypeError:                              # older torch without the dynamo option
    torch.onnx.export(model, torch.zeros(1, 3, 256, 256), ONNX, **kwargs)
print(f"exported {ONNX.name} · {ONNX.stat().st_size/1e6:.1f} MB")

# check: original vs exported on 600 training tiles
X   = np.load(f"{TRN}/X.npy", mmap_mode="r")
ytr = np.load(f"{TRN}/y.npy")
idx = np.sort(np.random.default_rng(0).choice(len(X), 600, replace=False))
x   = torch.from_numpy(np.ascontiguousarray(X[idx])).permute(0, 3, 1, 2).float().div(255)

with torch.no_grad():

    ref = model(x).numpy()
sess = ort.InferenceSession(str(ONNX), providers=["CPUExecutionProvider"])
out  = np.concatenate([sess.run(None, {"tiles": x[i:i+64].numpy()})[0] for i in range(0, len(x), 64)])

diff = float(np.abs(ref - out).max())
print(f"largest difference, original vs ONNX: {diff:.2e}")
print(f"mean score · egg tiles {out[ytr[idx] == 1].mean():.3f} · empty tiles {out[ytr[idx] == 0].mean():.3f}")
assert diff < 1e-4, "the exported model does not match the original"

/tmp/ipykernel_23/2332821324.py:21: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.zeros(1, 3, 256, 256), ONNX, dynamo=False, **kwargs)


exported detector.onnx · 44.7 MB
largest difference, original vs ONNX: 2.03e-06
mean score · egg tiles 0.818 · empty tiles 0.044


## 3 · The diagnosis dataset

The 65 patients' images are downloaded again from Zenodo (record 6467268), with
the same parallel downloader and checksum check as notebook 04. They are needed
to cut the tile crops for the review screen, and to test the exported model
end to end on real samples.

In [3]:
import random, shutil, hashlib, threading, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def pick_scratch():
    for cand in ("/kaggle/temp", "/tmp"):
        p = pathlib.Path(cand)
        try:
            p.mkdir(parents=True, exist_ok=True)
            (p / ".w").write_text("x"); (p / ".w").unlink()
            return p
        except Exception:
            pass
    raise RuntimeError("no scratch dir")

SCRATCH  = pick_scratch()
ZIP_PATH = SCRATCH / "diagnosis_test_dataset.zip"

rec  = requests.get("https://zenodo.org/api/records/6467268", timeout=60).json()
meta = next(f for f in rec["files"] if f["key"] == "diagnosis_test_dataset.zip")
SRC_URL, TOTAL = meta["links"]["self"], meta["size"]
MD5_PUB = meta["checksum"].split(":")[1]

N_CONN, RETRIES, CHUNK = 16, 8, 4 * 1024 * 1024
with open(ZIP_PATH, "wb") as f:
    f.truncate(TOTAL)
bar, lock = tqdm(total=TOTAL, unit="B", unit_scale=True, unit_divisor=1024, desc="download"), threading.Lock()

def fetch_range(start, end):
    err = None
    for attempt in range(RETRIES):
        pos = start

        try:
            with requests.get(SRC_URL, headers={"Range": f"bytes={pos}-{end}"},
                              stream=True, timeout=(30, 300)) as r:
                if r.status_code != 206:
                    raise RuntimeError(f"expected 206, got {r.status_code}")
                with open(ZIP_PATH, "r+b") as f:
                    f.seek(pos)
                    for chunk in r.iter_content(chunk_size=CHUNK):
                        if chunk:
                            f.write(chunk); pos += len(chunk)
                            with lock: bar.update(len(chunk))
            if pos - 1 == end:
                return True, None
            raise RuntimeError("short segment")
        except Exception as e:
            with lock: bar.update(-(pos - start))
            time.sleep(min(2 ** attempt, 30) + random.random() * 2)
            err = f"{type(e).__name__}: {e}"
    return False, err

seg    = TOTAL // N_CONN
ranges = [(i * seg, TOTAL - 1 if i == N_CONN - 1 else (i + 1) * seg - 1) for i in range(N_CONN)]
with ThreadPoolExecutor(N_CONN) as ex:
    for fut in as_completed([ex.submit(fetch_range, s, e) for s, e in ranges]):
        ok, err = fut.result()
        assert ok, f"segment failed: {err}"
bar.close()

md5 = hashlib.md5()
with open(ZIP_PATH, "rb") as f:
    for b in iter(lambda: f.read(64 * 1024 * 1024), b""):
        md5.update(b)

assert md5.hexdigest() == MD5_PUB, "checksum mismatch: file is corrupt"
print(f"verified · {TOTAL/1e9:.2f} GB · md5 {MD5_PUB}")

download:   0%|          | 0.00/11.7G [00:00<?, ?B/s]

verified · 12.52 GB · md5 76c20a6c5f566c2f141b6dd274ea4d74


### Matching the images to the saved scores

The archive is checked for the expected structure, and its patients are lined
up with the scores saved by notebook 04. Every patient must appear in both, in
the same order, with frames named the same way, so that each saved tile score
points at the right piece of the right image.

In [4]:
zf    = zipfile.ZipFile(ZIP_PATH)
IMGS  = [n for n in zf.namelist() if n.lower().endswith(".jpeg")]
by_id = collections.defaultdict(dict)
for n in IMGS:
    p = pathlib.PurePosixPath(n)
    by_id[p.parent.name][p.name] = n
assert len(by_id) == 65 and all(len(v) == 117 for v in by_id.values()), "unexpected structure"

z       = np.load(SCORES)
S, y    = z["S"], z["y"]
SAMPLES = [str(s) for s in z["samples"]]
FRAMES  = z["frames"].astype(str)

assert set(SAMPLES) == set(by_id), "patients differ between archive and scores"
for i, sid in enumerate(SAMPLES):
    assert set(FRAMES[i]) == set(by_id[sid]), f"{sid}: frame names differ"
print(f"65 patients · 117 frames each · scores {S.shape} line up with the images")

65 patients · 117 frames each · scores (65, 117, 12) line up with the images


## 4 · Tiles for the review screen

For each of the 65 patients, the twelve highest-scoring tiles, ranked by the
scores saved in notebook 04, are cut from the original images with exactly the
same resize and grid as everywhere else (1024 × 768, 3 × 4 tiles of 256 × 256)
and saved as JPEGs. A manifest records each tile's rank, score, source image and
position, so the backend can serve the reviewer the actual regions the detector
ranked highest.

In [5]:
import pandas as pd
from PIL import Image

ROWS, COLS, PX = 3, 4, 256

def tiles_of(data):
    a = np.asarray(Image.open(io.BytesIO(data)).convert("RGB")
                     .resize((1024, 768), Image.BILINEAR))
    return (a.reshape(ROWS, PX, COLS, PX, 3)
             .transpose(0, 2, 1, 3, 4)
             .reshape(ROWS * COLS, PX, PX, 3))

TILE_DIR = WORK / "tiles"
manifest = {}
for i, sid in enumerate(tqdm(SAMPLES, desc="cutting tiles")):
    flat_scores = S[i].reshape(-1)
    top = np.argsort(-flat_scores, kind="stable")[:12]
    (TILE_DIR / sid).mkdir(parents=True, exist_ok=True)
    cache, entries = {}, []
    for rank, flat in enumerate(top, 1):
        f, t  = divmod(int(flat), 12)
        frame = FRAMES[i, f]
        if frame not in cache:
            cache[frame] = tiles_of(zf.read(by_id[sid][frame]))
        name = f"{rank:02d}.jpg"
        Image.fromarray(cache[frame][t]).save(TILE_DIR / sid / name, quality=90)
        entries.append(dict(rank=rank, score=round(float(flat_scores[flat]), 4),
                            image=frame.rsplit(".", 1)[0], tile=t + 1, file=f"{sid}/{name}"))
    manifest[sid] = entries

with open(WORK / "tiles_manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=1)


size = sum(p.stat().st_size for p in TILE_DIR.rglob("*.jpg"))
print(f"{sum(len(v) for v in manifest.values())} tiles saved · {size/1e6:.1f} MB")
print("DT40 top 3:", [(e["score"], e["image"], e["tile"]) for e in manifest["DT40"][:3]])

cutting tiles:   0%|          | 0/65 [00:00<?, ?it/s]

780 tiles saved · 10.8 MB
DT40 top 3: [(0.9927, '0004_0004', 12), (0.9917, '0009_0003', 7), (0.9668, '0006_0004', 7)]


## 5 · The exported model, end to end

The real test of the export: every patient's 117 images are tiled and scored
again, this time by the ONNX model on the processor, exactly as the backend will
do it, and compared with the scores saved by notebook 04 (PyTorch on a GPU). Small
numerical differences are expected from the change of hardware; what matters is
whether any patient's confident-tile count, and with it the triage decision,
changes. The time per patient estimates how long one uploaded sample will take
to analyse.

In [6]:
def onnx_scores(tiles):                           # uint8 (N, 256, 256, 3) → scores (N,)
    x = tiles.transpose(0, 3, 1, 2).astype(np.float32) / 255.0
    return np.concatenate([sess.run(None, {"tiles": x[j:j+64]})[0] for j in range(0, len(x), 64)])

def decide(count, clear_below, flag_above):
    return "negative" if count < clear_below else ("positive" if count > flag_above else "referred")

rows = []
with ThreadPoolExecutor(4) as ex:
    for i, sid in enumerate(tqdm(SAMPLES, desc="scoring with ONNX")):
        t0    = time.time()
        raw   = [zf.read(by_id[sid][fr]) for fr in FRAMES[i]]
        tiles = np.stack(list(ex.map(tiles_of, raw))).reshape(-1, PX, PX, 3)
        s     = onnx_scores(tiles)
        ref   = S[i].reshape(-1)
        rows.append(dict(id=sid, secs=time.time() - t0, maxdiff=float(np.abs(s - ref).max()),
                         onnx95=int((s > 0.95).sum()), saved95=int((ref > 0.95).sum()),
                         onnx99=int((s > 0.99).sum()), saved99=int((ref > 0.99).sum())))

df = pd.DataFrame(rows)
df["safety_same"]     = [decide(a, 1, 35) == decide(b, 1, 35) for a, b in zip(df.onnx95, df.saved95)]
df["efficiency_same"] = [decide(a, 1, 12) == decide(b, 1, 12) for a, b in zip(df.onnx99, df.saved99)]

print(f"largest score difference, any tile : {df.maxdiff.max():.4f}")
print(f"patients whose count changed       : {(df.onnx95 != df.saved95).sum()} at 0.95 · "
      f"{(df.onnx99 != df.saved99).sum()} at 0.99")
print(f"patients whose decision changed    : {(~df.safety_same).sum()} Safety · "
      f"{(~df.efficiency_same).sum()} Efficiency")
print(f"time per patient on this CPU       : median {df.secs.median():.0f} s · max {df.secs.max():.0f} s")
changed = df[(df.onnx95 != df.saved95) | (df.onnx99 != df.saved99)]
if len(changed):
    print("\n", changed[["id", "saved95", "onnx95", "saved99", "onnx99", "safety_same", "efficiency_same"]].to_string(index=False))

scoring with ONNX:   0%|          | 0/65 [00:00<?, ?it/s]

largest score difference, any tile : 0.0120
patients whose count changed       : 3 at 0.95 · 2 at 0.99
patients whose decision changed    : 0 Safety · 0 Efficiency
time per patient on this CPU       : median 33 s · max 55 s

   id  saved95  onnx95  saved99  onnx99  safety_same  efficiency_same
DT10       23      22        8       8         True             True
DT13       22      21       21      21         True             True
DT18       59      59       39      40         True             True
DT29      241     240      193     193         True             True
DT42       17      17        7       6         True             True


## 6 · The deployed rule and the app bundle

The triage rule the app uses is fitted once on all 65 patients, with the same
procedure as notebook 04 section 9, for both the Efficiency and the Safety
setting, and saved to `triage_config.json`. These values configure the app; the
performance figures remain the cross-validated ones from notebook 04. The tile
crops and their manifest are packed into one archive for the backend.

In [7]:
T    = S.reshape(65, -1)
THRS = (0.9, 0.95, 0.99, 0.995, 0.999)
C    = {t: (T > t).sum(1) for t in THRS}

def fit_rule(margin=None):
    best = None
    for t in THRS:
        c, pmin = C[t], C[t][y == 1].min()
        hi = int(C[t][y == 0].max())
        if margin is None:
            lo = int(pmin)
        else:
            if pmin < margin:
                continue
            lo = 1
        cov = ((c < lo) | (c > hi)).mean()
        if best is None or cov > best[0]:
            best = (cov, t, lo, hi)
    _, t, lo, hi = best
    return dict(tile_threshold=t, clear_below=lo, flag_above=hi)

config = {
    "description": "Bilharz triage rule fitted on all 65 patients (notebook 05). A tile is confident "
                   "if its detector score exceeds tile_threshold; a sample is negative if its "
                   "confident-tile count is below clear_below, positive if above flag_above, "
                   "otherwise referred.",
    "default": "safety",
    "settings": {"efficiency": fit_rule(), "safety": fit_rule(margin=2)},
    "model": {"file": "detector.onnx", "input": "float32 (N, 3, 256, 256), values 0–1",
              "image_resize": [1024, 768], "grid": [3, 4], "tile_px": 256},
}
with open(WORK / "triage_config.json", "w") as fh:

    json.dump(config, fh, indent=2)
for name, r in config["settings"].items():
    print(f"{name:10s} threshold {r['tile_threshold']} · negative if count < {r['clear_below']} "
          f"· positive if count > {r['flag_above']}")

with zipfile.ZipFile(WORK / "tiles.zip", "w", zipfile.ZIP_STORED) as zt:
    for p in sorted(TILE_DIR.rglob("*.jpg")):
        zt.write(p, p.relative_to(TILE_DIR))
    zt.write(WORK / "tiles_manifest.json", "tiles_manifest.json")
shutil.rmtree(TILE_DIR)                       # keep only the archive in the output
print(f"\ntiles.zip · {(WORK / 'tiles.zip').stat().st_size/1e6:.1f} MB")
print("output:", sorted(p.name for p in WORK.iterdir()))

efficiency threshold 0.99 · negative if count < 1 · positive if count > 12
safety     threshold 0.95 · negative if count < 1 · positive if count > 35

tiles.zip · 11.0 MB
output: ['__notebook__.ipynb', 'detector.onnx', 'tiles.zip', 'tiles_manifest.json', 'triage_config.json']


## 7 · Summary

- **The detector is exported to ONNX** (44.7 MB), with normalisation and the
  sigmoid built in. On 600 training tiles it matches the original to within
  0.000002.
- **End to end on all 65 patients**, scored on the processor as the backend will
  do it, the exported model **reproduces every triage decision** under both
  settings. Confident-tile counts differ by one tile in 5 patients, from the
  half-precision GPU scoring of notebook 04.
- **Scoring one sample takes about 34 seconds** on Kaggle's processor (at most
  61), the expected cost of analysing an upload.
- **The review screen gets real tiles**: the twelve highest-scoring tiles for
  each patient, 780 in all, with a manifest of their ranks, scores and positions.
- **The deployed rule**, fitted on all 65 patients: Efficiency clears at 0 tiles
  above 0.99 and confirms above 12; Safety clears at 0 tiles above 0.95 and
  confirms above 35.

The output holds `detector.onnx`, `triage_config.json`, `tiles_manifest.json` and
`tiles.zip`, everything the backend needs.